In [1]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from datasets import load_dataset
import os
from huggingface_hub import login
from google.colab import userdata

In [2]:
import transformers
transformers.logging.set_verbosity_info()

In [3]:
os.environ['TQDM_DISABLE'] = 'false'

In [4]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [8]:
# Конфигурация
MODEL_NAME = "sberbank-ai/rugpt3medium_based_on_gpt2"  # базовая модель для дообучения ruGPT3
#MODEL_NAME = "sberbank-ai/rugpt3small_based_on_gpt2"
DATASET_NAME = "nikrog/psychology_dataset_rus"  # датасет
FINETUNED_MODEL_NAME = "nikrog/rugpt3medium_finetuned_psychology"
OUTPUT_DIR = "./rugpt3_finetuned"

# Параметры LoRA/QLoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["c_attn"]  # для GPT-2 архитектуры

# QLoRA настройки
USE_QLORA = False # True для QLoRA, False для обычного LoRA
LOAD_IN_4BIT = True
LOAD_IN_8BIT = False

In [6]:
def load_model_and_tokenizer():
    """Загрузка модели и токенизатора"""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    if USE_QLORA:
        # Конфигурация для 4-битного квантования
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=LOAD_IN_4BIT,
            load_in_8bit=LOAD_IN_8BIT,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=quantization_config,
            device_map="auto",
            trust_remote_code=True
        )

        # Подготовка модели для k-bit training
        model = prepare_model_for_kbit_training(model)
    else:
        # Обычная загрузка для LoRA
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            device_map="auto",
            torch_dtype=torch.float16
        )

    return model, tokenizer

def configure_lora(model):
    """Настройка LoRA адаптеров"""
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    return model


def load_and_preprocess_dataset(tokenizer, max_length=512):
    """Загрузка и предобработка датасета"""

    data_files = {
        "train": "train.txt",
        "test": "test.txt",
        "val": "val.txt"
    }

    # Загрузка датасета из Hugging Face
    dataset = load_dataset(DATASET_NAME, data_files=data_files, token=hf_token)
    dataset_train = dataset["train"]
    dataset_val = dataset["val"]
    dataset_test = dataset["test"]

    print(f"Train примеров: {len(dataset['train'])}")
    print(f"Val примеров: {len(dataset['val'])}")
    print(f"Test примеров: {len(dataset['test'])}")

    def preprocess_function(examples):
        # Токенизация текста
        tokenized = tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length,
            padding="max_length"
        )
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    # Применение предобработки
    processed_train_dataset = dataset_train.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset_train.column_names
    )

    processed_val_dataset = dataset_val.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset_val.column_names
    )

    processed_test_dataset = dataset_test.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset_test.column_names
    )

    # Разделение на train/eval
    #split_dataset = processed_dataset.train_test_split(test_size=0.1)
    #return split_dataset["train"], split_dataset["test"]

    return processed_train_dataset, processed_val_dataset, processed_test_dataset

def train_model(model, tokenizer, train_dataset, eval_dataset):
    """Обучение модели"""
    # Data collator
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

    # Параметры обучения
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=8 #4,
        per_device_eval_batch_size=8 #4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=1, #3,
        weight_decay=0.01,
        fp16=True, # only for GPU
        logging_steps=50,
        save_strategy="epoch",
        eval_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
        gradient_checkpointing=True if USE_QLORA else False,
        ## for google colab
        disable_tqdm=False,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
    )

    # Инициализация Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )

    # Запуск обучения
    trainer.train()

    # Сохранение модели
    #trainer.save_model(OUTPUT_DIR)
    #tokenizer.save_pretrained(OUTPUT_DIR)

    return trainer, tokenizer

def save_model_to_hf(trainer, tokenizer):
    """Сохранение модели"""
    # Загрузка на Hugging Face
    trainer.push_to_hub(
        repo_id=FINETUNED_MODEL_NAME,
        private=True,
        commit_message="Load fine-tuned model on psychology dataset",
    )

    # Токенизатор тоже нужно загрузить
    tokenizer.push_to_hub(
        repo_id=FINETUNED_MODEL_NAME,
        private=True,
        commit_message="Load tokenizer for fine-tuned model"
    )


In [9]:
print("Загрузка модели и токенизатора...")
model, tokenizer = load_model_and_tokenizer()

Загрузка модели и токенизатора...


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--sberbank-ai--rugpt3medium_based_on_gpt2/snapshots/d7fd75dc7644c87ee6b147040464a9b950715bdc/config.json
Model config GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 1,
  "embd_pdrop": 0.1,
  "eos_token_id": 2,
  "id2label": {
    "0": "LABEL_0"
  },
  "initializer_range": 0.02,
  "label2id": {
    "LABEL_0": 0
  },
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 2048,
  "n_embd": 1024,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 24,
  "n_positions": 2048,
  "n_special": 0,
  "output_past": true,
  "pad_token_id": 0,
  "predict_special_tokens": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to

pytorch_model.bin:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--sberbank-ai--rugpt3medium_based_on_gpt2/snapshots/d7fd75dc7644c87ee6b147040464a9b950715bdc/pytorch_model.bin
Generate config GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 2,
  "output_attentions": false,
  "output_hidden_states": false,
  "pad_token_id": 0,
  "use_cache": true
}

We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sberbank-ai/rugpt3medium_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Generation config file not found, using a generation config created from the model config.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--sberbank-ai--rugpt3medium_based_on_gpt2/snapshots/d7fd75dc7644c87ee6b147040464a9b950715bdc/config.json
Generate config Generati

In [10]:
print("Настройка LoRA адаптеров...")
model = configure_lora(model)

Настройка LoRA адаптеров...
trainable params: 1,572,864 || all params: 408,907,776 || trainable%: 0.3847


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [11]:
print("Загрузка и предобработка датасета...")
train_dataset, eval_dataset, test_dataset = load_and_preprocess_dataset(tokenizer)

Загрузка и предобработка датасета...


README.md:   0%|          | 0.00/723 [00:00<?, ?B/s]

train.txt:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

test.txt:   0%|          | 0.00/2.69M [00:00<?, ?B/s]

val.txt:   0%|          | 0.00/3.14M [00:00<?, ?B/s]

...

Generating val split: 0 examples [00:00, ? examples/s]

Train примеров: 128904
Val примеров: 16112
Test примеров: 16114


Map:   0%|          | 0/128904 [00:00<?, ? examples/s]

Map:   0%|          | 0/16112 [00:00<?, ? examples/s]

Map:   0%|          | 0/16114 [00:00<?, ? examples/s]

In [12]:
print(f"Train примеров: {len(train_dataset)}")
print(f"Val примеров: {len(eval_dataset)}")
print(f"Test примеров: {len(test_dataset)}")

Train примеров: 128904
Val примеров: 16112
Test примеров: 16114


In [13]:
print("Начало обучения...")
trainer, tokenizer = train_model(model, tokenizer, train_dataset, eval_dataset)
print("Обучение завершено!")


PyTorch: setting up devices


Начало обучения...


***** Running training *****
  Num examples = 128,904
  Num Epochs = 3
  Instantaneous batch size per device = 4
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 4
  Total optimization steps = 24,171
  Number of trainable parameters = 1,572,864
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
save_model_to_hf(trainer, tokenizer)
print(f"Модель сохранена в: {FINETUNED_MODEL_NAME}")

In [ ]:
# 1. Проверка, что модель обучена
print(trainer.state.global_step)  # Должно быть > 0

# 2. Проверка доступа к весам
print(trainer.model)  # Должна отображаться архитектура модели

# 3. Тестовая генерация (пример)
input_text = "Психология — это"
inputs = tokenizer(input_text, return_tensors="pt").to(trainer.model.device)
outputs = trainer.model.generate(**inputs, max_length=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))